# SDP and second order cone program for quantum MaxCut

In [1]:
using LinearAlgebra
import Graphs
import IterTools
#using BenchmarkTools

include("SwapMat.jl")
include("QMC.jl")

function GroundStateEnergyMaxCut(A)
    return LinearAlgebra.eigmin(HamiltonianMaxCut(A); permute=true, scale=true)
end

Clarabel not installed


GroundStateEnergyMaxCut (generic function with 1 method)

In [9]:
#G = Graphs.cycle_graph(n)
#G = Graphs.barabasi_albert(n, 5)
#G = Graphs.erdos_renyi(n,0.5)

In [37]:
solver = "mosek"

# two edges
A = [0 1 0 0; 
     1 0 0 0;
     0 0 0 1;
     0 0 1 0]
#cycle
A2 = [0 1 0 1; 
     1 0 1 0;
     0 1 0 1;
     1 0 1 0]
# clique
A3= [0 1 1 1;   
     1 0 1 1;
     1 1 0 1;
     1 1 1 0]
QMC_SOC(A,  pauli_NPA1=true,  four_qb=true,  VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver)

(-1.999999999771029,   [1, 2]  =  -1.0
  [1, 3]  =  0.5
  [1, 4]  =  0.5
  [2, 3]  =  0.5
  [2, 4]  =  0.5
  [3, 4]  =  -1.0)

In [ ]:
#cylce
solver = "mosek"
A = [0 1 0 1; 
     1 0 1 0;
     0 1 0 1;
     1 0 1 0]
QMC_SOC(A,  pauli_NPA1=true,  four_qb=true,  VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver)

## small graphs

In [3]:
#test different constraints
n = 16#28
G = Graphs.erdos_renyi(n,0.2)
A = Graphs.adjacency_matrix(G)
nE = sum(A) / 2
solver = "mosek"

val1, X = QMC_SOC(A,  pauli_NPA1=false, four_qb=false, VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver);
val2, X = QMC_SOC(A,  pauli_NPA1=false, four_qb=true,  VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver);

val3, X = QMC_SOC(A,  pauli_NPA1=true,  four_qb=false, VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver);
val4, X = QMC_SOC(A,  pauli_NPA1=true,  four_qb=true,  VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver);

val6, X = QMC_SOC(A,  pauli_NPA1=false, four_qb=true,  VarBench_norm=false, verbose=false, partition_31=false, partition_31_relax=false, solver=solver);
val8, X = QMC_SOC(A,  pauli_NPA1=true,  four_qb=true,  VarBench_norm=false, verbose=false, partition_31=false, partition_31_relax=false, solver=solver);

println("number edge : ", nE)

println("      all irreps    no [3,1] block \n ")
print("no NPA \n")
println("3 qb   ", round(val1, digits=5))
println("4 qb   ", round(val2, digits=5), "   ", round(val6, digits=5), "\n")

print("with NPA \n")
println("3 qb   ", round(val3, digits=5))
println("4 qb   ", round(val4, digits=5), "   ", round(val8, digits=5))

number edge : 37.0
      all irreps    no [3,1] block 
 
no NPA 
3 qb   -21.33343
4 qb   -20.34548   -21.33342

with NPA 
3 qb   -20.20356
4 qb   -19.7437   -20.20355


## timing

In [6]:
solver = "mosek"

@time QMC_SOC(A,  pauli_NPA1=false, four_qb=false, VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver);
@time QMC_SOC(A,  pauli_NPA1=false, four_qb=true,  VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver);

print("\n with NPA \n")
@time QMC_SOC(A,  pauli_NPA1=true,  four_qb=false, VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver);
@time QMC_SOC(A,  pauli_NPA1=true,  four_qb=true,  VarBench_norm=false, verbose=false, partition_31=true, partition_31_relax=false, solver=solver);

print("\n no partition 31 \n")
@time QMC_SOC(A,  pauli_NPA1=false, four_qb=true,  VarBench_norm=false, verbose=false, partition_31=false, partition_31_relax=false, solver=solver);
@time QMC_SOC(A,  pauli_NPA1=true,  four_qb=true,  VarBench_norm=false, verbose=false, partition_31=false, partition_31_relax=false, solver=solver);

  0.037408 seconds (202.05 k allocations: 10.125 MiB)
  1.893130 seconds (6.78 M allocations: 403.390 MiB, 19.86% gc time)

 with NPA 
  0.116005 seconds (225.18 k allocations: 11.163 MiB)
  2.093517 seconds (6.85 M allocations: 406.057 MiB, 18.67% gc time)

 no partition 31 
  0.865412 seconds (5.02 M allocations: 267.794 MiB, 6.46% gc time)
  1.383667 seconds (5.09 M allocations: 270.493 MiB, 31.20% gc time)


In [ ]:
#test different constraints
# test SOC vs SOC+SDP for random graphs
# high connectivity p = 0.8: roughly SOC = 2.1  * SOC+SDP
# low connectivity  p = 0.2: roughly SOC = 1.02 * SOC+SDP

s = zeros(5)
m = 1

n = 40

for t=1:5
    p = t/5
    println(p)
    for i=1:m
        G = Graphs.erdos_renyi(n,p)
        A = Graphs.adjacency_matrix(G)
        nE = sum(A) / 2
        
        val1, X = QMC_SOC(A, pauli_NPA1=false,VarBench_norm=true, verbose=false);
        val2, X = QMC_SOC(A, pauli_NPA1=true, VarBench_norm=true, verbose=false);
        #if round(nE + val1, digits=3) != 0
        #    s = s+1 #print(A)
        #end
        #println(" ", nE)
        #println(round(val1, digits=8))
        #println(round(val2, digits=8))
        #println(" ")
        s[t] = s[t] + val1/val2 
    end
    s[t] = s[t]/m
    println(s[t])
end
round.(s, digits=2)

# 3-RDM approach: $\binom{n}{3}$ variable SOCP

## 50 qubits

In [ ]:
# timing on 50 qubits

G = Graphs.erdos_renyi(50,0.4)
A = Graphs.adjacency_matrix(G)
@time QMC_SOC(A; pauli_NPA1=false, starbound=false, starbound4=false)
@time QMC_SOC(A; pauli_NPA1=false, starbound=false, starbound4=true)
@time QMC_SOC(A; pauli_NPA1=true,  starbound=false, starbound4=false)
@time QMC_SOC(A; pauli_NPA1=true,  starbound=false, starbound4=true)

## 100 qubits

In [ ]:
G = Graphs.erdos_renyi(100,0.4)
A = Graphs.adjacency_matrix(G)
@time QMC_SOC(A; pauli_NPA1=true, starbound=false, starbound4=false)